# Brain Stroke Deep Learning: Advanced Fine-Tuning & Real-World Preprocessing

This notebook provides the complete, state-of-the-art training and inference pipeline for:
1. **Real-World Image Preprocessing**: Automatic skull stripping, artifact cleaning, CLAHE contrast enhancement, and radiologist Hounsfield Unit (HU) windowing for DICOM.
2. **Two-Stage Fine-Tuning**: Head warmup + deep convolutional fine-tuning with cosine learning rate decay.
3. **Balanced Class Weighting**: Solves the severe ischemic stroke class imbalance.
4. **Grad-CAM Explainability (XAI)**: High-resolution visual heatmaps.
5. **Direct Model Export**: Saves `.keras` models ready for the Flask application (`app.py`).

In [ ]:
# Install required medical imaging libraries
!pip install -q pydicom opencv-python-headless scikit-learn kagglehub

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPUs Available: {tf.config.list_physical_devices('GPU')}")

## 1. Real-World Image Preprocessing Pipeline
Real-world CT and MRI scans come in DICOM (`.dcm`) format or images with patient text, scanner borders, and varying contrasts.
Here we implement:
- **DICOM Hounsfield Unit (HU) Windowing**: Standard Brain Window (Center = 40 HU, Width = 80 HU).
- **Skull / Head Localization**: Crops away background annotations and dark scanner borders.
- **CLAHE Contrast Enhancement**: Accentuates faint ischemic hypodensities and hemorrhagic hyperdensities.
- **Letterbox Resizing**: Preserves exact anatomical aspect ratios.

In [ ]:
import pydicom
from PIL import Image, ImageOps

def load_raw_scan(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == '.dcm':
        ds = pydicom.dcmread(file_path)
        pix = ds.pixel_array.astype(np.float32)
        slope = float(getattr(ds, 'RescaleSlope', 1.0))
        intercept = float(getattr(ds, 'RescaleIntercept', 0.0))
        hu = pix * slope + intercept
        # Standard Brain Window: Center 40, Width 80
        w_min, w_max = 40.0 - 40.0, 40.0 + 40.0
        clipped = np.clip(hu, w_min, w_max)
        norm = ((clipped - w_min) / (w_max - w_min) * 255.0).astype(np.uint8)
        return cv2.cvtColor(norm, cv2.COLOR_GRAY2RGB)
    else:
        with Image.open(file_path) as img:
            img = ImageOps.exif_transpose(img)
            return np.array(img.convert('RGB'))

def clean_and_crop_head(img_rgb, margin=0.03):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img_rgb
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < (img_rgb.shape[0] * img_rgb.shape[1] * 0.15):
        return img_rgb
    x, y, w, h = cv2.boundingRect(c)
    px, py = int(w * margin), int(h * margin)
    x1, y1 = max(0, x - px), max(0, y - py)
    x2, y2 = min(img_rgb.shape[1], x + w + px), min(img_rgb.shape[0], y + h + py)
    return img_rgb[y1:y2, x1:x2]

def enhance_contrast_clahe(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enh = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l_enh, a, b)), cv2.COLOR_LAB2RGB)

def letterbox_resize(img_rgb, target_size=(224, 224)):
    th, tw = target_size
    h, w = img_rgb.shape[:2]
    scale = min(tw / w, th / h)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(img_rgb, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((th, tw, 3), dtype=np.uint8)
    top, left = (th - nh) // 2, (tw - nw) // 2
    canvas[top:top+nh, left:left+nw] = resized
    return canvas

def preprocess_pipeline(file_path, target_size=(224, 224)):
    raw = load_raw_scan(file_path)
    cropped = clean_and_crop_head(raw)
    enhanced = enhance_contrast_clahe(cropped)
    return letterbox_resize(enhanced, target_size)

## 2. Model Architecture with Medical Augmentations & Official ImageNet Normalization
We build a transfer learning model that uses:
1. Official `preprocess_input` (preventing the double-rescaling bug).
2. Safe medical augmentations (horizontal symmetry flip, slight head tilt $\pm 14^\circ$, subtle contrast jitter).
3. Dropout and L2 regularization to prevent overfitting on small clinical cohorts.

In [ ]:
def build_stroke_model(num_classes=3, img_size=(224, 224), backbone='MobileNetV2'):
    inputs = tf.keras.Input(shape=(*img_size, 3), name='scan_input')
    
    # Safe medical augmentations
    x = tf.keras.layers.RandomFlip('horizontal')(inputs)
    x = tf.keras.layers.RandomRotation(0.04)(x)
    x = tf.keras.layers.RandomBrightness(0.08)(x)
    x = tf.keras.layers.RandomContrast(0.08)(x)
    
    # Model-specific ImageNet preprocessing
    if backbone == 'MobileNetV2':
        x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
        base = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_tensor=x)
    elif backbone == 'DenseNet121':
        x = tf.keras.applications.densenet.preprocess_input(x)
        base = tf.keras.applications.DenseNet121(include_top=False, weights='imagenet', input_tensor=x)
    else:
        raise ValueError(f'Unknown backbone: {backbone}')
        
    base.trainable = False
    
    # Classification head
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(base.output)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    x = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)
    
    return tf.keras.Model(inputs=inputs, outputs=outputs), base

## 3. Two-Stage Fine-Tuning Execution (MRI Scan)
We run:
- **Stage 1 (Head Warmup)**: Trains classifier head for 5 epochs with Adam ($10^{-3}$).
- **Stage 2 (Deep Fine-Tuning)**: Unfreezes top 30 convolutional layers with Cosine Decay learning rate ($5 \times 10^{-5}$ down to $10^{-6}$).
- **Class Weighting**: Automatically balances Ischemic vs Hemorrhagic vs Normal samples.

In [ ]:
# Dataset Path (Adjust to your local directory or Kaggle path)
MRI_DIR = r'C:\Users\ASUS\Downloads\archive\dataset\Stroke_classification'

train_ds = tf.keras.utils.image_dataset_from_directory(
    MRI_DIR, validation_split=0.2, subset='training', seed=42,
    image_size=(224, 224), batch_size=16, shuffle=True
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    MRI_DIR, validation_split=0.2, subset='validation', seed=42,
    image_size=(224, 224), batch_size=16, shuffle=True
)
class_names = train_ds.class_names
print(f"Classes: {class_names}")

# Compute balanced weights
y_all = [label for _, labels in train_ds for label in labels.numpy()]
weights = compute_class_weight('balanced', classes=np.unique(y_all), y=y_all)
class_weights = dict(zip(np.unique(y_all), weights))
print(f"Balanced Weights: {class_weights}")

train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

model, base = build_stroke_model(num_classes=len(class_names), backbone='MobileNetV2')

# Stage 1
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=5, class_weight=class_weights)

# Stage 2: Deep Fine-Tuning
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]
model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weights, callbacks=callbacks)

# Save the fine-tuned model
model.save('stroke_cnn_model_finetuned.keras')
print('[OK] Fine-tuned model saved successfully!')

## 4. Evaluation: Precision, Recall, and Confusion Matrix

In [ ]:
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, labels=list(range(len(class_names))), target_names=class_names, digits=4))
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

## 5. Grad-CAM Visualization with Preprocessed Images

In [ ]:
def get_last_conv_layer(m):
    for layer in reversed(m.layers):
        if isinstance(layer, tf.keras.Model):
            for sub in reversed(layer.layers):
                if isinstance(sub, tf.keras.layers.Conv2D):
                    return sub
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer
    return None

def generate_gradcam(preprocessed_img, model):
    last_conv = get_last_conv_layer(model)
    grad_model = tf.keras.models.Model([model.inputs], [last_conv.output, model.output])
    inp = np.expand_dims(preprocessed_img, axis=0)
    
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(inp)
        pred_idx = tf.argmax(preds[0])
        loss = preds[:, pred_idx]
        
    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    
    heatmap = cv2.resize(heatmap.numpy(), (224, 224))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(cv2.cvtColor(preprocessed_img, cv2.COLOR_RGB2BGR), 0.6, heatmap_color, 0.4, 0)
    return cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB), int(pred_idx)

# Example test on a sample image
sample_img = val_ds.take(1)
for img_batch, _ in sample_img:
    test_scan = np.uint8(img_batch[0].numpy())
    overlay, p_idx = generate_gradcam(test_scan, model)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(test_scan)
    plt.title("Input Scan")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title(f"Grad-CAM Explanation: {class_names[p_idx]}")
    plt.axis('off')
    plt.show()
    break